<a href="https://colab.research.google.com/github/alinsonbo-debug/agenticai1/blob/main/BizOps_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip -q install crewai flask requests
import os, re, json, sqlite3, random, threading, datetime as dt
from collections import Counter
import requests
from flask import Flask, request, jsonify, g
from werkzeug.serving import make_server

DB_PATH = "aiops_bizops.db"
API = "http://127.0.0.1:5000"
random.seed(42)

In [27]:
SCHEMA = """
DROP TABLE IF EXISTS agent_task_logs; DROP TABLE IF EXISTS reports;
DROP TABLE IF EXISTS customer_feedback; DROP TABLE IF EXISTS sales_transactions; DROP TABLE IF EXISTS agents;

CREATE TABLE agents (
  agent_id TEXT PRIMARY KEY, role TEXT NOT NULL, permissions TEXT NOT NULL,
  token TEXT NOT NULL UNIQUE, active INTEGER NOT NULL DEFAULT 1);

CREATE TABLE sales_transactions (
  id INTEGER PRIMARY KEY AUTOINCREMENT, txn_date TEXT NOT NULL, product TEXT NOT NULL, region TEXT NOT NULL,
  units INTEGER NOT NULL CHECK (units > 0), unit_price REAL NOT NULL, unit_cost REAL NOT NULL,
  revenue REAL NOT NULL, cost REAL NOT NULL);

CREATE TABLE customer_feedback (
  id INTEGER PRIMARY KEY AUTOINCREMENT, fb_date TEXT NOT NULL, customer_id TEXT NOT NULL, product TEXT NOT NULL,
  rating INTEGER CHECK (rating BETWEEN 1 AND 5), comment TEXT NOT NULL,
  sentiment_score REAL, keywords TEXT);

CREATE TABLE reports (
  id INTEGER PRIMARY KEY AUTOINCREMENT, created_at TEXT NOT NULL, period_start TEXT NOT NULL, period_end TEXT NOT NULL,
  total_revenue REAL NOT NULL, gross_margin_pct REAL NOT NULL, avg_sentiment REAL NOT NULL,
  body TEXT NOT NULL, status TEXT NOT NULL DEFAULT 'DRAFT', author TEXT, approved_by TEXT);

CREATE TABLE agent_task_logs (
  id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT NOT NULL, agent_id TEXT, task TEXT, endpoint TEXT,
  outcome TEXT, detail TEXT);

CREATE INDEX idx_sales_date ON sales_transactions(txn_date);
CREATE INDEX idx_fb_date ON customer_feedback(fb_date);
"""

# Identity registry: least-privilege permissions per agent
AGENTS = {
  "researcher": ("Research Agent",  "read:sales,read:feedback",                 "tok-researcher-7f3a"),
  "analyst":    ("Analysis Agent",  "read:sales,read:feedback",                 "tok-analyst-91bc"),
  "reporter":   ("Reporting Agent", "read:sales,read:feedback,write:reports",   "tok-reporter-22de"),
  "validator":  ("Validation Agent","read:reports,read:sales",                  "tok-validator-5a10"),
  "manager":    ("Management Agent","read:reports,approve:reports,read:logs",   "tok-manager-c8e4"),
}
TOKENS = {k: v[2] for k, v in AGENTS.items()}

POS = {"great","fast","excellent","love","friendly","helpful","reliable","smooth","good","easy","amazing","perfect"}
NEG = {"late","slow","broken","damaged","terrible","rude","expensive","bad","refund","delay","missing","poor","angry"}
STOP = {"the","a","an","and","is","was","it","to","of","my","for","very","but","on","in","i","with","this","that","so"}

def sentiment(text):
    w = re.findall(r"[a-z]+", text.lower())
    p, n = sum(x in POS for x in w), sum(x in NEG for x in w)
    return round((p - n) / max(p + n, 1), 2)

def keywords(text, k=3):
    w = [x for x in re.findall(r"[a-z]+", text.lower()) if x not in STOP and len(x) > 3]
    return ",".join(x for x, _ in Counter(w).most_common(k))

def seed(days=60):
    con = sqlite3.connect(DB_PATH)
    con.executescript(SCHEMA)
    for aid, (role, perms, tok) in AGENTS.items():
        con.execute("INSERT INTO agents VALUES (?,?,?,?,1)", (aid, role, perms, tok))
    products = {"Retail Subscription": (49, 20), "Logistics Tracking": (89, 45), "Express Delivery": (25, 17), "Analytics Add-on": (120, 40)}
    regions = ["UK", "EU", "US"]
    today = dt.date.today()
    good = ["great fast delivery", "excellent support team friendly", "love the smooth reliable service", "easy and helpful"]
    bad = ["late delivery terrible", "slow support rude", "damaged parcel want refund", "missing items poor service delay"]
    for d in range(days, -1, -1):
        day = today - dt.timedelta(days=d)
        for _ in range(random.randint(8, 14)):
            prod = random.choice(list(products)); price, cost = products[prod]
            units = random.randint(1, 12)
            if prod == "Express Delivery" and d < 14: units = max(1, units // 2)   # planted trend: sales dip
            con.execute("INSERT INTO sales_transactions (txn_date,product,region,units,unit_price,unit_cost,revenue,cost) VALUES (?,?,?,?,?,?,?,?)",
                        (str(day), prod, random.choice(regions), units, price, cost, round(units*price,2), round(units*cost,2)))
        for _ in range(random.randint(2, 5)):
            prod = random.choice(list(products))
            unhappy = prod == "Express Delivery" and d < 14 and random.random() < 0.7   # planted trend: complaints
            c = random.choice(bad if unhappy else good + bad[:1])
            con.execute("INSERT INTO customer_feedback (fb_date,customer_id,product,rating,comment,sentiment_score,keywords) VALUES (?,?,?,?,?,?,?)",
                        (str(day), f"C{random.randint(1000,1999)}", prod, random.randint(1,2) if c in bad else random.randint(3,5), c, sentiment(c), keywords(c)))
    con.commit(); con.close()

seed()
con = sqlite3.connect(DB_PATH)
for t in ["sales_transactions","customer_feedback","agents"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows")
con.close()

sales_transactions 674 rows
customer_feedback 210 rows
agents 5 rows


In [28]:
import pandas as pd
con = sqlite3.connect(DB_PATH)
display(pd.read_sql("SELECT * FROM sales_transactions LIMIT 5", con))
display(pd.read_sql("SELECT * FROM customer_feedback LIMIT 5", con))
display(pd.read_sql("SELECT agent_id, role, permissions FROM agents", con))
con.close()

,id,txn_date,product,region,units,unit_price,unit_cost,revenue,cost
0,1,2026-09-11,Retail Subscription,US,1,49.0,20.0,49.0,20.0
1,2,2026-09-11,Express Delivery,UK,4,25.0,17.0,100.0,68.0
2,3,2026-09-11,Logistics Tracking,UK,12,89.0,45.0,1068.0,540.0
3,4,2026-09-11,Retail Subscription,EU,10,49.0,20.0,490.0,200.0
4,5,2026-09-11,Retail Subscription,UK,1,49.0,20.0,49.0,20.0


,id,fb_date,customer_id,product,rating,comment,sentiment_score,keywords
0,1,2026-09-11,C1549,Retail Subscription,3,easy and helpful,1.0,"easy,helpful"
1,2,2026-09-11,C1565,Analytics Add-on,4,great fast delivery,1.0,"great,fast,delivery"
2,3,2026-09-11,C1196,Express Delivery,1,late delivery terrible,-1.0,"late,delivery,terrible"
3,4,2026-09-11,C1791,Retail Subscription,4,excellent support team friendly,1.0,"excellent,support,team"
4,5,2026-09-12,C1224,Express Delivery,2,late delivery terrible,-1.0,"late,delivery,terrible"


,agent_id,role,permissions
0,researcher,Research Agent,"read:sales,read:feedback"
1,analyst,Analysis Agent,"read:sales,read:feedback"
2,reporter,Reporting Agent,"read:sales,read:feedback,write:reports"
3,validator,Validation Agent,"read:reports,read:sales"
4,manager,Management Agent,"read:reports,approve:reports,read:logs"


In [29]:
import pandas as pd

con = sqlite3.connect(DB_PATH)

tables = ["sales_transactions", "customer_feedback", "reports", "agent_task_logs", "agents"]
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", con)
    df.to_csv(f"{t}.csv", index=False)
    print(f"Saved {t}.csv  ({len(df)} rows)")

con.close()

Saved sales_transactions.csv  (674 rows)
Saved customer_feedback.csv  (210 rows)
Saved reports.csv  (0 rows)
Saved agent_task_logs.csv  (0 rows)
Saved agents.csv  (5 rows)


In [30]:
from google.colab import files
for t in ["sales_transactions", "customer_feedback", "reports", "agent_task_logs", "agents"]:
    files.download(f"{t}.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
con = sqlite3.connect(DB_PATH)
with open("aiops_bizops_dump.sql", "w") as f:
    for line in con.iterdump():
        f.write(line + "\n")
con.close()
print("Saved aiops_bizops_dump.sql")

Saved aiops_bizops_dump.sql


In [32]:
app = Flask(__name__)

def db():
    if "db" not in g:
        g.db = sqlite3.connect(DB_PATH); g.db.row_factory = sqlite3.Row
    return g.db

@app.teardown_appcontext
def close_db(_):
    c = g.pop("db", None)
    if c: c.close()

def log(agent, task, outcome, detail=""):
    db().execute("INSERT INTO agent_task_logs (ts,agent_id,task,endpoint,outcome,detail) VALUES (?,?,?,?,?,?)",
                 (dt.datetime.now().isoformat(timespec="seconds"), agent, task, request.path, outcome, detail))
    db().commit()

def requires(perm):
    def deco(fn):
        def wrapper(*a, **kw):
            row = db().execute("SELECT * FROM agents WHERE token=? AND active=1", (request.headers.get("X-Agent-Token", ""),)).fetchone()
            if not row:
                log(None, fn.__name__, "DENIED", "bad token"); return jsonify(error="unauthenticated"), 401
            if perm not in row["permissions"].split(","):
                log(row["agent_id"], fn.__name__, "DENIED", f"missing {perm}"); return jsonify(error="forbidden", need=perm), 403
            g.agent = row["agent_id"]; log(g.agent, fn.__name__, "OK")
            return fn(*a, **kw)
        wrapper.__name__ = fn.__name__
        return wrapper
    return deco

In [33]:
def window(days, offset=0):
    end = dt.date.today() - dt.timedelta(days=offset)
    return str(end - dt.timedelta(days=days)), str(end)   # (start exclusive, end inclusive)

def sales_totals(days, offset=0):
    s, e = window(days, offset)
    r = db().execute("SELECT COALESCE(SUM(revenue),0) rev, COALESCE(SUM(cost),0) cost, COUNT(*) n FROM sales_transactions WHERE txn_date>? AND txn_date<=?", (s, e)).fetchone()
    return dict(revenue=round(r["rev"],2), cost=round(r["cost"],2), margin_pct=round(100*(r["rev"]-r["cost"])/r["rev"],2) if r["rev"] else 0, transactions=r["n"])

@app.get("/api/sales/summary")
@requires("read:sales")
def sales_summary():
    days = int(request.args.get("days", 7)); s, e = window(days)
    by_prod = {}
    for p in [r[0] for r in db().execute("SELECT DISTINCT product FROM sales_transactions")]:
        cur = db().execute("SELECT COALESCE(SUM(revenue),0) FROM sales_transactions WHERE product=? AND txn_date>? AND txn_date<=?", (p, s, e)).fetchone()[0]
        ps, pe = window(days, days)
        prev = db().execute("SELECT COALESCE(SUM(revenue),0) FROM sales_transactions WHERE product=? AND txn_date>? AND txn_date<=?", (p, ps, pe)).fetchone()[0]
        by_prod[p] = dict(current=round(cur,2), previous=round(prev,2), change_pct=round(100*(cur-prev)/prev,1) if prev else None)
    daily = [dict(r) for r in db().execute("SELECT txn_date d, ROUND(SUM(revenue),2) revenue FROM sales_transactions WHERE txn_date>? AND txn_date<=? GROUP BY txn_date ORDER BY txn_date", (s, e))]
    return jsonify(period=[s, e], current=sales_totals(days), previous=sales_totals(days, days), by_product=by_prod, daily=daily)

@app.get("/api/feedback/summary")
@requires("read:feedback")
def feedback_summary():
    days = int(request.args.get("days", 7)); s, e = window(days)
    rows = db().execute("SELECT * FROM customer_feedback WHERE fb_date>? AND fb_date<=?", (s, e)).fetchall()
    kw = Counter(k for r in rows for k in (r["keywords"] or "").split(",") if k)
    by = {}
    for r in rows: by.setdefault(r["product"], []).append(r["sentiment_score"])
    return jsonify(count=len(rows), avg_sentiment=round(sum(r["sentiment_score"] for r in rows)/len(rows),3) if rows else 0,
                   negative=sum(r["sentiment_score"] < 0 for r in rows),
                   by_product={p: round(sum(v)/len(v),2) for p, v in by.items()}, top_keywords=kw.most_common(8))

In [34]:
@app.post("/api/reports")
@requires("write:reports")
def create_report():
    b = request.get_json()
    cur = db().execute("INSERT INTO reports (created_at,period_start,period_end,total_revenue,gross_margin_pct,avg_sentiment,body,status,author) VALUES (?,?,?,?,?,?,?,'DRAFT',?)",
                       (dt.datetime.now().isoformat(timespec="seconds"), b["period_start"], b["period_end"], b["total_revenue"], b["gross_margin_pct"], b["avg_sentiment"], b["body"], g.agent))
    db().commit(); return jsonify(id=cur.lastrowid, status="DRAFT"), 201

@app.get("/api/reports")
@requires("read:reports")
def list_reports():
    return jsonify([dict(r) for r in db().execute("SELECT * FROM reports ORDER BY id DESC")])

@app.get("/api/reports/<int:rid>/reconcile")
@requires("read:reports")
def reconcile(rid):
    """Independent re-computation from the raw ledger. Compares to the figures the Reporter stored."""
    r = db().execute("SELECT * FROM reports WHERE id=?", (rid,)).fetchone()
    if not r: return jsonify(error="not found"), 404
    x = db().execute("SELECT COALESCE(SUM(revenue),0) rev, COALESCE(SUM(cost),0) c FROM sales_transactions WHERE txn_date>? AND txn_date<=?", (r["period_start"], r["period_end"])).fetchone()
    true_rev = round(x["rev"], 2); true_m = round(100*(x["rev"]-x["c"])/x["rev"], 2) if x["rev"] else 0
    ok = abs(true_rev - r["total_revenue"]) < 0.01 and abs(true_m - r["gross_margin_pct"]) < 0.01
    return jsonify(ok=ok, reported=dict(revenue=r["total_revenue"], margin=r["gross_margin_pct"]), ledger=dict(revenue=true_rev, margin=true_m))

@app.post("/api/reports/<int:rid>/decision")
@requires("approve:reports")
def decide(rid):
    status = request.get_json()["status"]
    if status not in ("APPROVED", "REJECTED"): return jsonify(error="invalid status"), 400
    db().execute("UPDATE reports SET status=?, approved_by=? WHERE id=?", (status, g.agent, rid)); db().commit()
    return jsonify(id=rid, status=status)

@app.get("/api/logs")
@requires("read:logs")
def logs():
    return jsonify([dict(r) for r in db().execute("SELECT * FROM agent_task_logs ORDER BY id DESC LIMIT 200")])

In [35]:
class Server(threading.Thread):
    def __init__(self): super().__init__(daemon=True); self.srv = make_server("127.0.0.1", 5000, app)
    def run(self): self.srv.serve_forever()
    def stop(self): self.srv.shutdown()

try: server.stop()
except Exception: pass
server = Server(); server.start()
print(requests.get(f"{API}/api/sales/summary", headers={"X-Agent-Token": TOKENS["researcher"]}).json()["current"])

INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "GET /api/sales/summary HTTP/1.1" 200 -


{'cost': 14577.0, 'margin_pct': 58.72, 'revenue': 35312.0, 'transactions': 75}


In [36]:
H = lambda agent: {"X-Agent-Token": TOKENS[agent]}

print("No token:        ", requests.get(f"{API}/api/reports").status_code)                                    # expect 401
print("Researcher write:", requests.post(f"{API}/api/reports", headers=H("researcher"), json={}).status_code)  # expect 403
print("Manager read log:", requests.get(f"{API}/api/logs", headers=H("manager")).status_code)                 # expect 200

INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "GET /api/reports HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "POST /api/reports HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "GET /api/logs HTTP/1.1" 200 -


No token:         401
Researcher write: 403
Manager read log: 200


In [37]:
def call(agent, method, path, **kw):
    r = requests.request(method, API + path, headers={"X-Agent-Token": TOKENS[agent]}, timeout=15, **kw)
    return r.status_code, r.json()

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

In [38]:
@tool("get_sales_summary")
def t_sales(days: int = 7) -> str:
    """Weekly sales, margin and per-product change from the finance database."""
    return json.dumps(call("researcher", "GET", f"/api/sales/summary?days={days}")[1])

@tool("get_feedback_summary")
def t_feedback(days: int = 7) -> str:
    """Customer sentiment summary and top keywords for the period."""
    return json.dumps(call("researcher", "GET", f"/api/feedback/summary?days={days}")[1])

@tool("save_draft_report")
def t_save(period_start: str, period_end: str, total_revenue: float, gross_margin_pct: float, avg_sentiment: float, body: str) -> str:
    """Save a DRAFT weekly report. Figures must come from the sales and feedback summaries."""
    return json.dumps(call("reporter", "POST", "/api/reports", json=dict(period_start=period_start, period_end=period_end,
                      total_revenue=total_revenue, gross_margin_pct=gross_margin_pct, avg_sentiment=avg_sentiment, body=body))[1])

@tool("reconcile_report")
def t_recon(report_id: int) -> str:
    """Compare a report's numbers to the raw ledger. Returns ok true or false."""
    return json.dumps(call("validator", "GET", f"/api/reports/{report_id}/reconcile")[1])

@tool("decide_report")
def t_decide(report_id: int, status: str) -> str:
    """Approve or reject a report. status is APPROVED or REJECTED."""
    return json.dumps(call("manager", "POST", f"/api/reports/{report_id}/decision", json=dict(status=status))[1])

In [39]:
def build_crew(llm=None):
    kw = dict(llm=llm) if llm else {}
    researcher = Agent(role="Research Agent", goal="Collect accurate sales and feedback data", backstory="Data analyst with read-only access. Never invents numbers.", tools=[t_sales, t_feedback], allow_delegation=False, **kw)
    analyst    = Agent(role="Analysis Agent", goal="Find the drivers behind revenue, margin and sentiment changes", backstory="Management accountant who explains variances.", allow_delegation=False, **kw)
    reporter   = Agent(role="Reporting Agent", goal="Write a clear weekly performance report and save it as a draft", backstory="Finance reporting specialist. Uses only figures supplied.", tools=[t_save], allow_delegation=False, **kw)
    validator  = Agent(role="Validation Agent", goal="Reconcile every reported figure to the ledger", backstory="Internal auditor. Rejects anything that does not tie out.", tools=[t_recon], allow_delegation=False, **kw)
    manager    = Agent(role="Management Agent", goal="Approve only reports that pass validation", backstory="Finance manager accountable for published reports.", tools=[t_decide], allow_delegation=False, **kw)
    tasks = [
      Task(description="Fetch the 7 day sales and feedback summaries.", expected_output="Raw figures as JSON.", agent=researcher),
      Task(description="Analyse variances vs the previous week. Name the worst product and the likely cause using feedback keywords.", expected_output="3 bullet insights.", agent=analyst),
      Task(description="Write the weekly report and save it as a draft. Return the report id and the figures.", expected_output="Report id.", agent=reporter),
      Task(description="Reconcile the draft report to the ledger.", expected_output="PASS or FAIL with figures.", agent=validator),
      Task(description="If validation is PASS call decide_report with APPROVED, otherwise REJECTED.", expected_output="Final status.", agent=manager),
    ]
    return Crew(agents=[researcher, analyst, reporter, validator, manager], tasks=tasks, process=Process.sequential)

In [40]:
print("Researcher reads sales:   ", call("researcher", "GET", "/api/sales/summary?days=7")[0])                      # expect 200
print("Researcher writes report: ", call("researcher", "POST", "/api/reports", json={})[0])                          # expect 403
print("Reporter approves report: ", call("reporter", "POST", "/api/reports/1/decision", json={"status": "APPROVED"})[0])  # expect 403
print("Validator reads reports:  ", call("validator", "GET", "/api/reports")[0])                                     # expect 200

INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "GET /api/sales/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "POST /api/reports HTTP/1.1" 403 -


Researcher reads sales:    200
Researcher writes report:  403


INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "POST /api/reports/1/decision HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:37] "GET /api/reports HTTP/1.1" 200 -


Reporter approves report:  403
Validator reads reports:   200


In [41]:
import asyncio

def run_deterministic():
    _, s = call("researcher", "GET", "/api/sales/summary?days=7")
    _, f = call("researcher", "GET", "/api/feedback/summary?days=7")
    cur, prev = s["current"], s["previous"]
    chg = round(100 * (cur["revenue"] - prev["revenue"]) / prev["revenue"], 1) if prev["revenue"] else 0
    worst = min(s["by_product"].items(), key=lambda kv: kv[1]["change_pct"] if kv[1]["change_pct"] is not None else 0)
    kws = ", ".join(k for k, _ in f["top_keywords"][:4])
    body = (f"WEEKLY PERFORMANCE REPORT {s['period'][0]} to {s['period'][1]}\n"
            f"Revenue: {cur['revenue']:,.2f} ({chg:+}% vs prior week). Gross margin: {cur['margin_pct']}% (prior {prev['margin_pct']}%).\n"
            f"Weakest product: {worst[0]} ({worst[1]['change_pct']}% revenue change).\n"
            f"Customer sentiment: {f['avg_sentiment']} across {f['count']} comments, {f['negative']} negative. Frequent themes: {kws}.\n"
            f"Action: review fulfilment and support for {worst[0]}.")
    code, r = call("reporter", "POST", "/api/reports", json=dict(period_start=s["period"][0], period_end=s["period"][1], total_revenue=cur["revenue"],
                   gross_margin_pct=cur["margin_pct"], avg_sentiment=f["avg_sentiment"], body=body))
    rid = r["id"]
    _, rec = call("validator", "GET", f"/api/reports/{rid}/reconcile")
    status = "APPROVED" if rec["ok"] else "REJECTED"
    call("manager", "POST", f"/api/reports/{rid}/decision", json=dict(status=status))
    return rid, status, rec

def run_pipeline():
    if os.environ.get("OPENAI_API_KEY"):
        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                # Execute the coroutine in the active event loop using nest_asyncio
                res = loop.run_until_complete(build_crew().kickoff_async())
                print(res)
            else:
                print(asyncio.run(build_crew().kickoff_async()))
        except Exception as e:
            print("CrewAI execution failed, falling back to deterministic:", e)
            rid, status, rec = run_deterministic()
            print(f"Report {rid}: {status}", rec)
    else:
        rid, status, rec = run_deterministic()
        print(f"Report {rid}: {status}", rec)

In [42]:
import pandas as pd
con = sqlite3.connect(DB_PATH)

# Run the pipeline first to populate the database tables if they are empty
try:
    run_pipeline()
except Exception as e:
    print("Could not run pipeline:", e)

print("=== LATEST REPORT ===")
df_reports = pd.read_sql("SELECT * FROM reports ORDER BY id DESC LIMIT 1", con)
if df_reports.empty:
    print("No reports found in the database. Please ensure run_pipeline() has successfully executed.")
else:
    r = df_reports.iloc[0]
    print(f"Report #{r['id']} | status: {r['status']} | author: {r['author']} | approved by: {r['approved_by']}\n")
    print(r["body"])

print("\n=== AUDIT TRAIL (last 12 actions) ===")
display(pd.read_sql("SELECT ts, agent_id, task, outcome FROM agent_task_logs ORDER BY id DESC LIMIT 12", con))
con.close()

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:39] "GET /api/sales/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:39] "GET /api/feedback/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:39] "POST /api/reports HTTP/1.1" 201 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:39] "GET /api/reports/1/reconcile HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:39] "POST /api/reports/1/decision HTTP/1.1" 200 -


CrewAI execution failed, falling back to deterministic: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
Report 1: APPROVED {'ledger': {'margin': 58.72, 'revenue': 35312.0}, 'ok': True, 'reported': {'margin': 58.72, 'revenue': 35312.0}}
=== LATEST REPORT ===
Report #1 | status: APPROVED | author: reporter | approved by: manager

WEEKLY PERFORMANCE REPORT 2026-11-03 to 2026-11-10
Revenue: 35,312.00 (+0.4% vs prior week). Gross margin: 58.72% (prior 58.21%).
Weakest product: Express Delivery (-23.3% revenue change).
Customer sentiment: 0.2 across 25 comments, 10 negative. Frequent themes: delivery, support, excellent, team.
Action: review fulfilment and support for Express Delivery.

=== AUDIT TRAIL (last 12 actions) ===


,ts,agent_id,task,outcome
0,2026-11-10T13:48:39,manager,decide,OK
1,2026-11-10T13:48:39,validator,reconcile,OK
2,2026-11-10T13:48:39,reporter,create_report,OK
3,2026-11-10T13:48:39,researcher,feedback_summary,OK
4,2026-11-10T13:48:39,researcher,sales_summary,OK
5,2026-11-10T13:48:37,validator,list_reports,OK
6,2026-11-10T13:48:37,reporter,decide,DENIED
7,2026-11-10T13:48:37,researcher,create_report,DENIED
8,2026-11-10T13:48:37,researcher,sales_summary,OK
9,2026-11-10T13:48:37,manager,logs,OK


In [43]:
!pip -q install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
run_pipeline()

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/sales/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/feedback/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "POST /api/reports HTTP/1.1" 201 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/reports/2/reconcile HTTP/1.1" 200 -


CrewAI execution failed, falling back to deterministic: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}


INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "POST /api/reports/2/decision HTTP/1.1" 200 -


Report 2: APPROVED {'ledger': {'margin': 58.72, 'revenue': 35312.0}, 'ok': True, 'reported': {'margin': 58.72, 'revenue': 35312.0}}


In [44]:
from IPython.display import HTML, display

def build_dashboard(path="dashboard.html"):
    _, s = call("reporter", "GET", "/api/sales/summary?days=14")
    _, f = call("reporter", "GET", "/api/feedback/summary?days=14")
    _, reps = call("manager", "GET", "/api/reports")
    _, lg = call("manager", "GET", "/api/logs")
    mx = max(d["revenue"] for d in s["daily"]) or 1
    bars = "".join(f'<div class="bar" title="{d["d"]}: {d["revenue"]}" style="height:{100*d["revenue"]/mx:.0f}%"></div>' for d in s["daily"])
    prod = "".join(f'<tr><td>{p}</td><td>{v["current"]:,.0f}</td><td class="{"neg" if (v["change_pct"] or 0)<0 else "pos"}">{v["change_pct"]}%</td><td>{f["by_product"].get(p,"-")}</td></tr>' for p, v in s["by_product"].items())
    rep = "".join(f'<tr><td>#{r["id"]}</td><td>{r["period_end"]}</td><td>{r["total_revenue"]:,.0f}</td><td>{r["status"]}</td><td>{r["approved_by"] or "-"}</td></tr>' for r in reps[:5])
    denied = sum(1 for l in lg if l["outcome"] == "DENIED")
    cur = s["current"]
    html = f"""<div style="font-family:system-ui,sans-serif;max-width:900px;margin:auto">
<style>.k{{display:flex;gap:12px}}.c{{flex:1;border:1px solid #ccc;border-radius:8px;padding:12px}}.c b{{font-size:22px;display:block}}
.bars{{display:flex;align-items:flex-end;gap:3px;height:120px;border-bottom:1px solid #999}}.bar{{flex:1;background:#2a9d8f}}
table{{width:100%;border-collapse:collapse;margin:8px 0 18px}}td,th{{border-bottom:1px solid #ddd;padding:6px;text-align:left}}.neg{{color:#c0392b}}.pos{{color:#1e8449}}</style>
<h2>Business Operations Dashboard</h2>
<div class="k"><div class="c">Revenue (14d)<b>{cur["revenue"]:,.0f}</b></div><div class="c">Gross margin<b>{cur["margin_pct"]}%</b></div>
<div class="c">Avg sentiment<b>{f["avg_sentiment"]}</b></div><div class="c">Negative comments<b>{f["negative"]}</b></div><div class="c">Denied agent calls<b>{denied}</b></div></div>
<h3>Daily revenue</h3><div class="bars">{bars}</div>
<h3>Products: revenue, change vs prior period, sentiment</h3><table><tr><th>Product</th><th>Revenue</th><th>Change</th><th>Sentiment</th></tr>{prod}</table>
<h3>Reports and approvals</h3><table><tr><th>ID</th><th>Period end</th><th>Revenue</th><th>Status</th><th>Approved by</th></tr>{rep}</table>
<h3>Top feedback keywords</h3><p>{", ".join(f"{k} ({n})" for k, n in f["top_keywords"])}</p></div>"""
    open(path, "w").write(html); return html

In [45]:
from google.colab import files

# Call the function to generate and write the dashboard.html file first
build_dashboard()

# Download the newly created dashboard.html
files.download("dashboard.html")

INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/sales/summary?days=14 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/feedback/summary?days=14 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/reports HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/logs HTTP/1.1" 200 -


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import unittest

class TestPlatform(unittest.TestCase):
    def test_unit_sentiment(self):
        self.assertGreater(sentiment("great fast service"), 0); self.assertLess(sentiment("late and damaged"), 0)
    def test_auth_required(self):
        self.assertEqual(requests.get(f"{API}/api/reports").status_code, 401)
    def test_least_privilege(self):
        self.assertEqual(call("researcher", "POST", "/api/reports", json={})[0], 403)
        self.assertEqual(call("reporter", "POST", "/api/reports/1/decision", json={"status": "APPROVED"})[0], 403)
    def test_denied_calls_are_logged(self):
        self.assertTrue(any(l["outcome"] == "DENIED" for l in call("manager", "GET", "/api/logs")[1]))
    def test_reconciliation_passes_for_genuine_report(self):
        rid, status, rec = run_deterministic(); self.assertTrue(rec["ok"]); self.assertEqual(status, "APPROVED")
    def test_validator_catches_wrong_figures(self):   # business KPI / agent behaviour validation
        s = call("reporter", "GET", "/api/sales/summary?days=7")[1]
        rid = call("reporter", "POST", "/api/reports", json=dict(period_start=s["period"][0], period_end=s["period"][1],
                   total_revenue=s["current"]["revenue"] + 500, gross_margin_pct=s["current"]["margin_pct"], avg_sentiment=0, body="tampered"))[1]["id"]
        self.assertFalse(call("validator", "GET", f"/api/reports/{rid}/reconcile")[1]["ok"])
    def test_db_integrity(self):
        c = sqlite3.connect(DB_PATH)
        self.assertEqual(c.execute("SELECT COUNT(*) FROM sales_transactions WHERE ABS(revenue-units*unit_price)>0.01").fetchone()[0], 0)

unittest.main(argv=["x"], exit=False, verbosity=2)

test_auth_required (__main__.TestPlatform.test_auth_required) ... INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/reports HTTP/1.1" 401 -
ok
test_db_integrity (__main__.TestPlatform.test_db_integrity) ... ok
test_denied_calls_are_logged (__main__.TestPlatform.test_denied_calls_are_logged) ... INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/logs HTTP/1.1" 200 -
ok
test_least_privilege (__main__.TestPlatform.test_least_privilege) ... INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "POST /api/reports HTTP/1.1" 403 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "POST /api/reports/1/decision HTTP/1.1" 403 -
ok
test_reconciliation_passes_for_genuine_report (__main__.TestPlatform.test_reconciliation_passes_for_genuine_report) ... INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/sales/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:47] "GET /api/feedback/summary?days=7 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/N

In [47]:
import pandas as pd
con = sqlite3.connect(DB_PATH)
display(pd.read_sql("SELECT id, period_end, total_revenue, status, author, approved_by FROM reports", con))
display(pd.read_sql("SELECT outcome, COUNT(*) AS calls FROM agent_task_logs GROUP BY outcome", con))
con.close()

,id,period_end,total_revenue,status,author,approved_by
0,1,2026-11-10,35312.0,APPROVED,reporter,manager
1,2,2026-11-10,35312.0,APPROVED,reporter,manager
2,3,2026-11-10,35312.0,APPROVED,reporter,manager
3,4,2026-11-10,35812.0,DRAFT,reporter,None


,outcome,calls
0,DENIED,7
1,OK,27


In [48]:
import pandas as pd
con = sqlite3.connect(DB_PATH)

for t in ["sales_transactions", "customer_feedback", "reports", "agent_task_logs", "agents"]:
    pd.read_sql(f"SELECT * FROM {t}", con).to_csv(f"{t}.csv", index=False)

with open("aiops_bizops_dump.sql", "w") as f:
    for line in con.iterdump():
        f.write(line + "\n")
con.close()

display(HTML(build_dashboard()))   # refresh dashboard.html with the latest data
print("Saved: 5 CSV files, aiops_bizops_dump.sql, dashboard.html")

INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:48] "GET /api/sales/summary?days=14 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:48] "GET /api/feedback/summary?days=14 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:48] "GET /api/reports HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2026 13:48:48] "GET /api/logs HTTP/1.1" 200 -


Saved: 5 CSV files, aiops_bizops_dump.sql, dashboard.html


In [49]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!rm -rf repo && git clone https://{token}@github.com/YOUR_USERNAME/bizops-agentic-ai.git repo
!cp aiops_bizops_dump.sql dashboard.html *.csv repo/
%cd repo
!git add .
!git commit -m "Update data exports"
!git push
%cd ..

Cloning into 'repo'...
remote: Repository not found.
fatal: repository 'https://github.com/YOUR_USERNAME/bizops-agentic-ai.git/' not found
cp: target 'repo/': No such file or directory
[Errno 2] No such file or directory: 'repo'
/
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
/


In [50]:
from google.colab import userdata
import requests
token = userdata.get("GH_TOKEN")
r = requests.get("https://api.github.com/user", headers={"Authorization": f"token {token}"})
print(r.status_code, r.json().get("login"))

200 alinsonbo-debug
